# CEP Sim MVP
Synthetic CEP memory simulator — Brazil Beer (Heineken) survey


In [1]:
import sys
sys.path.insert(0, '../../../..')  # project root

import logging
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s - %(message)s')


Matplotlib is building the font cache; this may take a moment.


## 1. Load config

In [2]:
from backend.schemas.config import load_cep_sim_config

config = load_cep_sim_config('../../../../backend/configs/cep_sim_config.toml')
print('Survey zip:', config.survey.zip_path)
print('Outputs dir:', config.output.outputs_dir)


Survey zip: backend/analysis/cep_sim/data/raw/DYNATA Raw Data CSV for Brazil and UK.zip
Outputs dir: outputs/cep_sim


## 2. Load raw survey

In [3]:
from backend.service.load_data import load_survey, inspect_survey

# Loads CSV from inside the Dynata zip archive
df = load_survey(config)
info = inspect_survey(df, config)
print(f"{info['row_count']} respondents, {info['recall_column_count']} recall columns")
print(f"CEP blocks: {info['cep_blocks']}")


FileNotFoundError: [Errno 2] No such file or directory: 'backend/analysis/cep_sim/data/raw/DYNATA Raw Data CSV for Brazil and UK.zip'

## 3. Reshape wide to long

In [ ]:
from backend.service.reshape_survey import reshape_wide_to_long, save_long_survey

# Parses codebook from zip to map Q10_1..Q20_21 columns → CEP description + brand name
long_df = reshape_wide_to_long(df, config)
save_long_survey(long_df, config)

print(f"Long table: {len(long_df):,} rows")
print(f"Mention rate: {long_df['mentioned'].mean():.1%}")
long_df.head()


## 4. Build CEP ontology

In [ ]:
from backend.service.ontology_builder import build_ontology, save_ontology

cep_master_df, raw_map_df = build_ontology(long_df, config)
save_ontology(cep_master_df, raw_map_df, config)

print(f"{len(cep_master_df)} CEPs across families:")
print(cep_master_df.groupby('cep_family').size().sort_values(ascending=False).to_string())
cep_master_df


## 5. Build respondent memory tables

In [ ]:
from backend.service.respondent_builder import (
    build_respondents, build_respondent_brand_cep, save_respondent_tables
)

respondents_df = build_respondents(df, config)
rbc_df = build_respondent_brand_cep(long_df, raw_map_df, config)
save_respondent_tables(respondents_df, rbc_df, config)

print(f"{len(respondents_df)} respondents")
print(f"{len(rbc_df):,} memory edges")

# Brand name lookup
brand_name_map = rbc_df[['brand_id','brand_name']].drop_duplicates().set_index('brand_id')['brand_name'].to_dict()

# Top brands by total recall mentions
top_brands = rbc_df.groupby('brand_name').size().sort_values(ascending=False).head(10)
print("\nTop 10 brands by recall edges:")
print(top_brands.to_string())


## 6. Baseline recall — single respondent example

In [ ]:
from backend.service.recall_engine import get_recall_probs, rank_brands, SCENARIOS

sample_id = str(respondents_df['respondent_id'].iloc[0])
scenario = SCENARIOS[0]  # trendy_bar

probs = get_recall_probs(sample_id, scenario['active_ceps'], rbc_df, cep_master_df, config)
ranked = rank_brands(probs)

print(f"Respondent {sample_id} | Scenario: {scenario['scenario_name']}")
print("Rank  Brand                   Prob")
for rank, (brand_id, prob) in enumerate(ranked[:8], 1):
    print(f"  {rank:2d}  {brand_name_map.get(brand_id, brand_id):<22}  {prob:.4f}")


## 7. Run baseline recall across all respondents × all scenarios

In [ ]:
from backend.service.validator import run_scenario_recall

respondent_ids = respondents_df['respondent_id'].astype(str).tolist()

scenario_recall_df = run_scenario_recall(
    respondent_ids, SCENARIOS, rbc_df, cep_master_df, brand_name_map, config
)

print(f"Scenario recall output: {len(scenario_recall_df):,} rows")

# Average recall prob by brand for trendy_bar
trendy_bar = scenario_recall_df[scenario_recall_df['scenario_name'] == 'trendy_bar']
avg_trendy_bar = trendy_bar.groupby('brand_name')['recall_prob'].mean().sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(9, 4))
avg_trendy_bar.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Avg Recall Probability — Trendy Bar')
ax.set_ylabel('Avg Recall Prob')
ax.set_xlabel('')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## 8. Define an ad and apply it

In [ ]:
from backend.service.ad_engine import Ad, apply_ad_to_population, save_episodic_events
from backend.service.ontology_builder import build_ontology

# Resolve CEP ids for the ad
trendy_bar_cep_id = cep_master_df[
    cep_master_df['cep_description'].str.contains('bar da moda', case=False, na=False)
]['cep_id'].values[0]

outdoor_hot_day_cep_id = cep_master_df[
    cep_master_df['cep_description'].str.contains('ao ar livre em um dia quente', case=False, na=False)
]['cep_id'].values[0]

# Define a Heineken ad targeting trendy bar + outdoor hot day
heineken_ad = Ad(
    ad_id='ad_heineken_trendy_bar_001',
    brand_id='brand_heineken',
    brand_name='Heineken',
    focal_ceps=[trendy_bar_cep_id],
    secondary_ceps=[outdoor_hot_day_cep_id],
    branding_clarity=0.9,
    attention_weight=1.0,
    channel='social_media',
    emotion='confidence',
)

print('Ad defined:', heineken_ad.ad_id)
print('Focal CEPs:', heineken_ad.focal_ceps)
print('Secondary CEPs:', heineken_ad.secondary_ceps)


In [ ]:
rbc_post, events = apply_ad_to_population(respondent_ids, heineken_ad, rbc_df, config)
save_episodic_events(events, config)

print(f"Applied ad to {len(events)} respondents")
print(f"RBC pre : {len(rbc_df):,} edges")
print(f"RBC post: {len(rbc_post):,} edges")


## 9. Compare recall before vs after

In [ ]:
from backend.service.validator import run_ad_impact, build_segment_summary

ad_impact_df = run_ad_impact(
    respondent_ids, SCENARIOS, rbc_df, rbc_post, cep_master_df, brand_name_map, config
)

# Show avg delta for trendy_bar scenario
trendy_bar_impact = (
    ad_impact_df[ad_impact_df['scenario_name'] == 'trendy_bar']
    .groupby('brand_name')[['recall_pre', 'recall_post', 'delta']]
    .mean()
    .round(4)
    .sort_values('delta', ascending=False)
    .head(8)
)
print("Avg recall shift — Trendy Bar")
print(trendy_bar_impact.to_string())


In [ ]:
# Visualise before/after for top brands
top8 = trendy_bar_impact.head(8).index
plot_df = trendy_bar_impact.loc[top8][['recall_pre', 'recall_post']]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(plot_df))
ax.bar([i - 0.2 for i in x], plot_df['recall_pre'],  width=0.4, label='Pre',  color='steelblue')
ax.bar([i + 0.2 for i in x], plot_df['recall_post'], width=0.4, label='Post', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df.index, rotation=35, ha='right')
ax.set_title('Recall Pre vs Post — Trendy Bar (Heineken ad)')
ax.set_ylabel('Avg Recall Score')
ax.legend()
plt.tight_layout()
plt.show()


## 10. Segment summary

In [ ]:
segment_summary_df = build_segment_summary(ad_impact_df, respondents_df)
print(f"Segment summary: {len(segment_summary_df):,} rows")
segment_summary_df[segment_summary_df['scenario_name'] == 'trendy_bar'].head(12)


## 11. Export all outputs

In [ ]:
from backend.service.validator import save_outputs, run_sanity_checks

paths = save_outputs(scenario_recall_df, ad_impact_df, segment_summary_df, config)
for name, path in paths.items():
    print(f"  {name}: {path}")

print("\n=== Sanity Checks ===")
checks = run_sanity_checks(scenario_recall_df, ad_impact_df)
for k, v in checks.items():
    print(f"  {k}: {v}")
